In [ ]:
# This notebook exists to answer two major questions. Primarily, the question of airports being structurally critical vs cosmetically large.
# Subsequently, is the EU air network efficient or just geographically lucky?


In [ ]:
import pandas as pd
import numpy as np
import networkx as nx
from math import radians, sin, cos, sqrt, atan2


In [ ]:
airports = pd.read_csv('/content/drive/MyDrive/airports.csv')
routes = pd.read_csv('/content/drive/MyDrive/routes.csv')
airport_route_counts = pd.read_csv('/content/drive/MyDrive/airport_route_counts.csv')

In [ ]:
airports.head(5)

,airport_iata,airport_icao,city,country,latitude,longitude,altitude
0,AAE,DABB,Annaba,Algeria,36.822201,7.809174,16
1,AAL,EKYT,Aalborg,Denmark,57.092759,9.849243,10
2,AAQ,URKA,Anapa,Russia,45.002102,37.347301,174
3,AAR,EKAH,Aarhus,Denmark,56.299999,10.619000,82
4,ABA,UNAA,Abakan,Russia,53.740002,91.385002,831


In [ ]:
routes.head(5)

,route_id,origin_airport,destination_airport,flights_per_day,flights_per_week,common_duration,price
0,46316,TZL,DTM,0-1 flights,4,125,70
1,46319,TZL,FMM,0-1 flights,4,95,30
2,46322,TZL,MLH,0-1 flights,4,110,40
3,77197,TZL,BER,1 flight,3,105,40
4,46323,TZL,MMX,1 flight,3,130,90


In [ ]:
airport_route_counts.head(5)

,airport_iata,route_count
0,AAL,25
1,AAR,21
2,ABZ,51
3,ACE,150
4,ADB,160


In [ ]:
airports.shape, routes.shape, airport_route_counts.shape

((960, 7), (18017, 7), (960, 2))

In [ ]:
G = nx.Graph()

for _, r in airports.iterrows():
    G.add_node(
        r["airport_iata"],
        lat=r["latitude"],
        lon=r["longitude"],
        country=r["country"]
    )

for _, r in routes.iterrows():
    G.add_edge(r["origin_airport"], r["destination_airport"])

In [ ]:
G.number_of_nodes(), G.number_of_edges()

(960, 11175)

In [ ]:
baseline_components = list(nx.connected_components(G))
baseline_component_count = len(baseline_components)
baseline_largest = max(len(c) for c in baseline_components)

In [ ]:
results = []

for airport in G.nodes():
    G_tmp = G.copy()
    G_tmp.remove_node(airport)

    components = list(nx.connected_components(G_tmp))
    largest = max(len(c) for c in components)

    results.append({
        "airport": airport,
        "component_increase": len(components) - baseline_component_count,
        "largest_component_loss": baseline_largest - largest
    })

impact = pd.DataFrame(results)

In [ ]:
impact.head()

,airport,component_increase,largest_component_loss
0,AAE,0,1
1,AAL,0,1
2,AAQ,0,1
3,AAR,0,1
4,ABA,0,1


In [ ]:
impact = impact.merge(
    airport_route_counts,
    left_on="airport",
    right_on="airport_iata",
    how="left"
)

In [ ]:
impact["damage_per_route"] = (
    impact["largest_component_loss"] / impact["route_count"]
)

In [ ]:
from math import radians, sin, cos, sqrt, atan2

def haversine(lat1, lon1, lat2, lon2):
    R = 6371
    lat1, lon1, lat2, lon2 = map(radians, [lat1, lon1, lat2, lon2])
    dlat = lat2 - lat1
    dlon = lon2 - lon1
    a = sin(dlat/2)**2 + cos(lat1)*cos(lat2)*sin(dlon/2)**2
    return 2 * R * atan2(sqrt(a), sqrt(1-a))

In [ ]:
def component_spread(G, nodes):
    coords = [
        (G.nodes[n]["lat"], G.nodes[n]["lon"])
        for n in nodes
    ]
    if len(coords) < 2:
        return 0
    dists = [
        haversine(lat1, lon1, lat2, lon2)
        for i,(lat1,lon1) in enumerate(coords)
        for lat2,lon2 in coords[i+1:]
    ]
    return np.mean(dists)

In [ ]:
largest_base_component = max(nx.connected_components(G), key=len)
baseline_spread = component_spread(G, largest_base_component)

In [ ]:
impact.columns

Index(['airport', 'component_increase', 'largest_component_loss',
       'airport_iata', 'route_count', 'damage_per_route'],
      dtype='object')

In [ ]:
impact["impact_norm"] = (
    impact["largest_component_loss"] /
    impact["largest_component_loss"].max()
)

In [ ]:
top_airports = impact.sort_values(
    "impact_norm", ascending=False
).head(20)["airport"]

In [ ]:
geo_results = []

for airport in top_airports:
    G_tmp = G.copy()
    G_tmp.remove_node(airport)

    largest = max(nx.connected_components(G_tmp), key=len)
    spread = component_spread(G_tmp, largest)

    geo_results.append({
        "airport": airport,
        "spread_increase": spread - baseline_spread
    })

geo_df = pd.DataFrame(geo_results)

In [ ]:
degree = dict(G.degree())
impact["degree"] = impact["airport"].map(degree)

impact["degree_norm"] = (
    impact["degree"] / impact["degree"].max()
)

In [ ]:
bet = nx.betweenness_centrality(G, normalized=True)
impact["betweenness"] = impact["airport"].map(bet)

In [ ]:
impact["final_score"] = (
    0.5 * impact["impact_norm"] +
    0.3 * impact["betweenness"] +
    0.2 * impact["degree_norm"]
)

In [ ]:
final_df = (
    impact
    .merge(
        airports[["airport_iata", "latitude", "longitude"]],
        left_on="airport",
        right_on="airport_iata",
        how="left"
    )
)

final_df = final_df[
    [
        "airport",
        "latitude",
        "longitude",
        "impact_norm",
        "betweenness",
        "degree_norm",
        "final_score"
    ]
]

In [ ]:
final_df.isna().sum()


,0
airport,0
latitude,0
longitude,0
impact_norm,0
betweenness,0
degree_norm,0
final_score,0


In [ ]:
final_df.to_csv("eu_airport_structural_scores.csv", index=False)